# Data Preparation — On-Policy Responses for a Llama-3.2-1B-Instruct Speculator

Speculative decoding pairs a large *target* (verifier) model with a small *draft* (speculator)
model: the draft proposes several tokens at once and the target verifies them in a single
forward pass, so accepted tokens come almost for free. The draft only helps if it predicts what
the target would actually have said, which means its training data has to be **on-policy** —
generated by the exact target model and generation config the speculator will be trained
against, not scraped from some other model's outputs.

This notebook produces that data. It runs the target model
(`unsloth/Llama-3.2-1B-Instruct`) over prompts from the **Open-PerfectBlend** dataset with vLLM,
writes the results as speculator-format JSONL (rows of `input_ids` and `loss_mask`), and pushes
the file to the Hugging Face Hub so the training notebook can pick it up.

Open-PerfectBlend is a broad instruction mixture (chat, math, code, reasoning) rather than the
single-generator prompts of Magpie, so the resulting draft sees a wider slice of the request
distribution it will face at serving time.

**Scope.** This notebook stops *before* `speculators prepare-data` (Step 1 of the upstream
tutorial). It covers environment setup and response regeneration only; tokenization into the
Arrow dataset, hidden-state generation, and DSpark training all happen in the training notebook.

**Environment.** Written for a Kaggle notebook with 2 GPUs (e.g. T4 x2) and internet enabled.

**Reference.**
[Speculators training tutorial](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train/)

## Step 0 — Set up the environment

The upstream tutorial recommends two separate virtual environments: `speculators_venv` for data
prep and training, and `vllm_venv` for serving the target model. That split exists to keep the
training stack from fighting vLLM over torch versions. Here both roles land in a single
environment, since this notebook only ever runs data prep, and the vLLM server it needs is
started and stopped by the regeneration script itself.

### Clone the speculators repository

The response-regeneration scripts live in-tree under `scripts/response_regeneration/` and are
not shipped in the published `speculators` wheel, so a clone is required rather than a plain
`pip install speculators`. The next cell `%cd`s into the checkout — every path in this notebook
is relative to the repository root from here on.

In [ ]:
!git clone https://github.com/vllm-project/speculators.git

In [ ]:
%cd speculators

### Install vLLM and speculators

| Command | Why |
| --- | --- |
| `uv venv speculators_venv` | Creates the venv the tutorial names, kept for parity with the docs. |
| `uv pip install "vllm>=0.22.0"` | Serves the target model and does the batched generation. |
| `uv pip install -e .` | Installs `speculators` from the clone, so the in-tree scripts are importable. |

Note the Kaggle caveat: each `!` line runs in its own shell, so `source speculators_venv/bin/activate`
does not carry over to the following lines or cells. The packages therefore end up in the
notebook's own Python environment, which is what the later cells actually import from. That is
fine here — it just means the venv is nominal.

In [ ]:
# Speculators venv (for data prep and training)
! uv venv speculators_venv
! source speculators_venv/bin/activate
! uv pip install "vllm>=0.22.0"
! uv pip install -e .

### Fix up the preinstalled Kaggle packages

Kaggle images ship a `torchaudio` build pinned to their own torch version, which conflicts with
the torch that vLLM pulls in and breaks the import chain; nothing here needs audio, so it is
simply removed. `datasets` is then upgraded because the Open-PerfectBlend loader needs a newer
version than the image provides — `--break-system-packages` is required to overwrite the
distribution-managed copy.

In [ ]:
! pip uninstall -y torchaudio
! pip install -Uq datasets --break-system-packages

## Step 1 — Response regeneration

Response regeneration is the step that makes the data on-policy: the target model is served with
vLLM and asked to answer prompts drawn from a seed dataset, and its own completions — not the
dataset's original answers — become the training targets. The output is written directly in
speculator format, one JSON object per line carrying `input_ids` and a `loss_mask` marking which
positions the draft is scored on.

`run_all.sh` orchestrates the whole thing: it starts a vLLM server (logging to `vllm_server.log`),
runs the regeneration over the dataset, and shuts the server down when it finishes.

| Flag | Meaning |
| --- | --- |
| `--model` | The target model to accelerate; the speculator is trained to imitate it. |
| `--dataset open-perfectblend` | Seed prompt source — a mixed-domain instruction blend. |
| `--limit 100000` | Number of samples to generate. |
| `--gpus 0,1` | GPU device IDs made visible to the server. |
| `--dp-size 2` | Data-parallel vLLM replicas — one per GPU, so both cards generate at once. |
| `--max-tokens 2048` | Cap on generated length per response; bounds the tail of long completions so the run fits in a session. |

Output: `open-perfectblend_Llama-3.2-1B-Instruct.jsonl` in the repository root.

On scale: the tutorial's 5K-sample runs are sanity checks that produce weak acceptance rates.
100K samples is a substantially larger run — expect it to dominate the session's wall clock, and
plan to resume or shard if the Kaggle time limit bites.

In [ ]:
! ./scripts/response_regeneration/run_all.sh \
  --model "unsloth/Llama-3.2-1B-Instruct" \
  --dataset open-perfectblend \
  --limit 100000 \
  --gpus 0,1 \
  --dp-size 2 \
  --max-tokens 2048

## Step 2 — Publish the dataset to the Hub

Kaggle sessions are ephemeral and training runs in a *separate* notebook, so the JSONL has to
outlive this session. Pushing it to the Hub makes it a stable, versioned input that the training
notebook can pull on any machine.

### Authenticate with Hugging Face

Reads the `HF_WRITE_YOSEFW` secret from Kaggle Secrets into `HF_TOKEN`, which `huggingface_hub`
picks up automatically. The secret must be attached to this notebook (Add-ons -> Secrets) and the
token needs **write** scope, or `create_repo` below will fail with a 403.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_WRITE_YOSEFW")

### Create the repo and upload

`create_repo(..., repo_type="dataset", exist_ok=True)` is idempotent, so re-running this cell
after a second generation pass just overwrites the file rather than erroring. `upload_file`
pushes the JSONL as-is; no reformatting is needed because it is already in speculator format.

Note the repo id below is `yosefw/magpie-llama-3.2-1b-instruct` — the same dataset repo the
Magpie run uses, with the Open-PerfectBlend JSONL landing beside it under its own filename.
Point it at a separate repo if you would rather keep the two mixtures apart.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo("yosefw/magpie-llama-3.2-1b-instruct", repo_type="dataset", exist_ok=True)
api.upload_file(
    path_or_fileobj="./open-perfectblend_Llama-3.2-1B-Instruct.jsonl",
    path_in_repo="open-perfectblend_Llama-3.2-1B-Instruct.jsonl",
    repo_id="yosefw/magpie-llama-3.2-1b-instruct",
    repo_type="dataset",
)

## Next steps

The dataset is now on the Hub. Training continues in the DSpark training notebooks —
`[A100] train-dspark-llama-3.2-1B-instruct-online.ipynb` or
`[2xT4 kaggle] train-dspark-llama-3.2-1B-instruct-online.ipynb` — which:

1. run `speculators prepare-data` to tokenize this JSONL into an Arrow dataset;
2. launch a vLLM server for `unsloth/Llama-3.2-1B-Instruct` to supply hidden states;
3. train the DSpark speculator in online mode (`--on-missing generate --on-generate delete`,
   so hidden states are produced on demand and never cached to disk);
4. serve the resulting checkpoint with `vllm serve` to measure acceptance length.